# RF / IQ Signal Classification Notebook

The idea is to keep the first version fairly simple: take raw complex I/Q captures, split them into fixed-length windows, and train a small CNN to classify the signal type. I have kept the raw I/Q approach because it lines up well with a lot of RF classification work, where the real and imaginary parts of the complex baseband signal are used directly rather than building a large set of manual features.

A few choices I am using here:

- raw I/Q samples are used directly as the model input
- each capture is split into overlapping windows so there are more training examples
- each window is represented as `(2, L)`, with I and Q as the two channels
- k-fold cross-validation is used so the result is less dependent on one train/test split
- groups are based on the original capture file, so windows from the same file do not leak between training and validation

This notebook is still meant as an experimental baseline. Once this is working reliably, the same structure should be adaptable to ADS-B captures and eventually transmitter ID classification by changing the dataset structure and labels.


In [ ]:
# Dummy I/Q data generator for a quick test dataset
import numpy as np
import os

# Basic settings for the generated test data
OUTPUT_DIR = "dataset_root"
CLASSES = ["bpsk", "qpsk", "fsk", "noise"]
NUM_SAMPLES_PER_CLASS = 50
SAMPLE_LENGTH = 2048  # number of IQ samples per capture
SNR_DB_RANGE = (0, 30)

os.makedirs(OUTPUT_DIR, exist_ok=True)

def add_awgn(signal, snr_db):
    signal_power = np.mean(np.abs(signal)**2)
    snr_linear = 10**(snr_db / 10)
    noise_power = signal_power / snr_linear
    noise = np.sqrt(noise_power/2) * (np.random.randn(*signal.shape) + 1j*np.random.randn(*signal.shape))
    return signal + noise

def generate_bpsk(n):
    bits = np.random.randint(0, 2, n)
    symbols = 2*bits - 1
    return symbols.astype(np.complex64)

def generate_qpsk(n):
    bits = np.random.randint(0, 4, n)
    mapping = {
        0: 1+1j,
        1: -1+1j,
        2: -1-1j,
        3: 1-1j
    }
    symbols = np.array([mapping[b] for b in bits]) / np.sqrt(2)
    return symbols.astype(np.complex64)

def generate_fsk(n):
    t = np.arange(n)
    f1, f2 = 0.05, 0.15
    bits = np.random.randint(0, 2, n)
    freq = np.where(bits == 0, f1, f2)
    phase = 2 * np.pi * np.cumsum(freq)
    return np.exp(1j * phase).astype(np.complex64)

def generate_noise(n):
    return (np.random.randn(n) + 1j*np.random.randn(n)).astype(np.complex64)

generators = {
    "bpsk": generate_bpsk,
    "qpsk": generate_qpsk,
    "fsk": generate_fsk,
    "noise": generate_noise
}

print("Generating dummy dataset...")

for cls in CLASSES:
    class_dir = os.path.join(OUTPUT_DIR, cls)
    os.makedirs(class_dir, exist_ok=True)

    for i in range(NUM_SAMPLES_PER_CLASS):
        signal = generators[cls](SAMPLE_LENGTH)

        if cls != "noise":
            snr_db = np.random.uniform(*SNR_DB_RANGE)
            signal = add_awgn(signal, snr_db)

        filepath = os.path.join(class_dir, f"{cls}_{i}.npy")
        np.save(filepath, signal)

print(f"Dataset created at: {OUTPUT_DIR}")

In [ ]:
# 1. Main configuration

from pathlib import Path

CONFIG = {
    # Point this at the dataset folder I want to use
    "dataset_root": Path("./dataset_root"),

    # Windowing setup
    "window_len": 128,          # common starting point for I/Q windows
    "stride": 64,               # overlap windows to get more examples
    "min_signal_len": 128,

    # Normalisation options
    "normalise_per_window": True,   # scale each window to unit power
    "demean": True,

    # Cross-validation settings
    "n_splits": 5,
    "random_seed": 42,

    # Training settings
    "batch_size": 256,
    "epochs": 20,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,

    # Model settings
    "dropout_conv": 0.25,
    "dropout_fc": 0.50,
    "num_filters_1": 50,
    "num_filters_2": 50,
    "kernel_size_1": 3,
    "kernel_size_2": 3,
    "fc_dim": 256,

    # Optional cap if the dataset gets too large
    "max_windows_per_class": None,   # for example, 10000

    # Output files
    "artifact_dir": Path("./artifacts"),
}

CONFIG["artifact_dir"].mkdir(parents=True, exist_ok=True)
CONFIG


In [ ]:
# 2. Imports

import json
import math
import random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["random_seed"])


## Loader and preprocessing

This part loads each capture, turns it into a complex vector, cuts it into fixed-length windows, and then converts each window into `(2, L)` format.

The two channels are just the I and Q components. I have kept that representation because it is easy to inspect and it matches the usual raw baseband setup used in RF classification examples.


In [ ]:
# 3. File loading and window generation

def load_iq_file(path: Path) -> np.ndarray:
    """
    Load one I/Q capture as a complex64 vector.
    This should cover the simple file formats I am likely to test first.
    """
    suffix = path.suffix.lower()

    if suffix == ".npy":
        arr = np.load(path, allow_pickle=False)
        if np.iscomplexobj(arr):
            iq = arr.astype(np.complex64).reshape(-1)
        else:
            arr = np.asarray(arr)
            if arr.ndim == 2 and arr.shape[1] == 2:
                iq = (arr[:, 0] + 1j * arr[:, 1]).astype(np.complex64)
            else:
                raise ValueError(f"Unsupported .npy format for {path}: shape={arr.shape}, dtype={arr.dtype}")

    elif suffix == ".npz":
        data = np.load(path, allow_pickle=False)
        keys = set(data.files)
        if "iq" in keys:
            arr = data["iq"]
            if np.iscomplexobj(arr):
                iq = arr.astype(np.complex64).reshape(-1)
            elif arr.ndim == 2 and arr.shape[1] == 2:
                iq = (arr[:, 0] + 1j * arr[:, 1]).astype(np.complex64)
            else:
                raise ValueError(f"Unsupported 'iq' array in {path}: shape={arr.shape}, dtype={arr.dtype}")
        elif {"I", "Q"}.issubset(keys):
            I = np.asarray(data["I"]).reshape(-1)
            Q = np.asarray(data["Q"]).reshape(-1)
            if len(I) != len(Q):
                raise ValueError(f"I and Q length mismatch in {path}")
            iq = (I + 1j * Q).astype(np.complex64)
        else:
            raise ValueError(f"Unsupported .npz structure for {path}; keys={sorted(keys)}")

    elif suffix in {".csv", ".txt"}:
        df = pd.read_csv(path)
        if df.shape[1] < 2:
            raise ValueError(f"Expected at least 2 columns in {path}")
        iq = (df.iloc[:, 0].to_numpy() + 1j * df.iloc[:, 1].to_numpy()).astype(np.complex64)

    else:
        raise ValueError(f"Unsupported file type: {path}")

    return iq


def normalise_iq(iq: np.ndarray, demean: bool = True) -> np.ndarray:
    iq = iq.astype(np.complex64, copy=False)
    if demean:
        iq = iq - np.mean(iq)
    power = np.mean(np.abs(iq) ** 2)
    if power > 0:
        iq = iq / np.sqrt(power + 1e-12)
    return iq.astype(np.complex64)


def iq_to_2ch(iq_window: np.ndarray) -> np.ndarray:
    return np.stack([iq_window.real, iq_window.imag], axis=0).astype(np.float32)


def generate_windows(iq: np.ndarray, window_len: int, stride: int):
    if len(iq) < window_len:
        return
    for start in range(0, len(iq) - window_len + 1, stride):
        yield iq[start:start + window_len]


def build_window_dataset(dataset_root: Path):
    rows = []
    class_dirs = [p for p in sorted(dataset_root.iterdir()) if p.is_dir()]
    if not class_dirs:
        raise FileNotFoundError(f"No class subfolders found under {dataset_root.resolve()}")

    for class_dir in class_dirs:
        label = class_dir.name
        files = [p for p in sorted(class_dir.rglob('*')) if p.is_file() and p.suffix.lower() in {'.npy', '.npz', '.csv', '.txt'}]

        print(f"[{label}] found {len(files)} files")
        added = 0

        for file_idx, path in enumerate(files):
            iq = load_iq_file(path)

            if len(iq) < CONFIG["min_signal_len"]:
                continue

            if not CONFIG["normalise_per_window"]:
                iq = normalise_iq(iq, demean=CONFIG["demean"])

            for win_idx, win in enumerate(generate_windows(iq, CONFIG["window_len"], CONFIG["stride"])):
                if CONFIG["normalise_per_window"]:
                    win = normalise_iq(win, demean=CONFIG["demean"])

                rows.append({
                    "x": iq_to_2ch(win),
                    "y": label,
                    "group": str(path),          # group by original file to prevent leakage
                    "file_name": path.name,
                    "class_name": label,
                })
                added += 1

                if CONFIG["max_windows_per_class"] is not None and added >= CONFIG["max_windows_per_class"]:
                    break

            if CONFIG["max_windows_per_class"] is not None and added >= CONFIG["max_windows_per_class"]:
                break

        print(f"  added {added} windows")

    if not rows:
        raise RuntimeError("No windows were created. Check the dataset path and file formats.")
    return rows


In [ ]:
# 4. Build the windowed dataset

rows = build_window_dataset(CONFIG["dataset_root"])

label_counts = Counter(r["y"] for r in rows)
print("Classes:", label_counts)
print("Total windows:", len(rows))

X = np.stack([r["x"] for r in rows])                      # windows as I/Q channels
y_text = np.array([r["y"] for r in rows])
groups = np.array([r["group"] for r in rows])

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)
class_names = list(label_encoder.classes_)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of groups/files:", len(np.unique(groups)))

pd.DataFrame({
    "class": class_names,
    "encoded": np.arange(len(class_names))
})


In [ ]:
# Quick plots to check the windows look sensible

def plot_example_per_class(X, y, class_names, ncols=1):
    n_classes = len(class_names)
    fig, axes = plt.subplots(n_classes, ncols, figsize=(12, 3 * n_classes), squeeze=False)
    for class_idx, class_name in enumerate(class_names):
        idx = np.where(y == class_idx)[0][0]
        ax = axes[class_idx, 0]
        ax.plot(X[idx, 0], label="I")
        ax.plot(X[idx, 1], label="Q")
        ax.set_title(f"{class_name} - example window")
        ax.legend()
    plt.tight_layout()
    plt.show()

plot_example_per_class(X, y, class_names)


## Model

This is a small 1D CNN using the two-channel I/Q input. I have kept it deliberately basic for now so it is easier to debug and compare results across datasets.

The model is:

- two convolution blocks
- max pooling
- dropout
- one fully connected layer
- final class output

For the current stage this is enough as a baseline. If the ADS-B dataset works well, the next step would be comparing this against deeper models or a transmitter-fingerprinting-specific setup.


In [ ]:
# 5. PyTorch dataset and model

class IQWindowDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class IQCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(2, CONFIG["num_filters_1"], kernel_size=CONFIG["kernel_size_1"], padding="same"),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(CONFIG["dropout_conv"]),

            nn.Conv1d(CONFIG["num_filters_1"], CONFIG["num_filters_2"], kernel_size=CONFIG["kernel_size_2"], padding="same"),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(CONFIG["dropout_conv"]),
        )

        reduced_len = CONFIG["window_len"] // 4
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(CONFIG["num_filters_2"] * reduced_len, CONFIG["fc_dim"]),
            nn.ReLU(),
            nn.Dropout(CONFIG["dropout_fc"]),
            nn.Linear(CONFIG["fc_dim"], num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

model = IQCNN(num_classes=len(class_names)).to(DEVICE)
print(model)
print("Trainable parameters:", count_parameters(model))


In [ ]:
# 6. Training helpers

def train_one_fold(model, train_loader, val_loader, epochs, learning_rate, weight_decay, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
    }

    best_state = None
    best_val_acc = -np.inf

    for epoch in range(1, epochs + 1):
        # Training pass
        model.train()
        running_loss = 0.0
        train_true, train_pred = [], []

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)
            preds = torch.argmax(logits, dim=1)
            train_true.extend(yb.detach().cpu().numpy())
            train_pred.extend(preds.detach().cpu().numpy())

        train_loss = running_loss / len(train_loader.dataset)
        train_acc = accuracy_score(train_true, train_pred)

        # Validation pass
        model.eval()
        running_loss = 0.0
        val_true, val_pred = [], []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)

                running_loss += loss.item() * xb.size(0)
                preds = torch.argmax(logits, dim=1)
                val_true.extend(yb.detach().cpu().numpy())
                val_pred.extend(preds.detach().cpu().numpy())

        val_loss = running_loss / len(val_loader.dataset)
        val_acc = accuracy_score(val_true, val_pred)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
        )

    model.load_state_dict(best_state)
    return model, history


@torch.no_grad()
def predict_model(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        logits = model(xb)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(yb.numpy())
    return np.array(y_true), np.array(y_pred)


## Cross-validation

For cross-validation I am splitting by original capture file rather than by window.

This matters because each capture produces many overlapping windows. If windows from the same file end up in both training and validation, the validation result can look better than it really is. Grouping by file gives a more realistic check of whether the model is learning something useful.


In [ ]:
# 7. Group-aware k-fold cross-validation

def make_group_stratified_folds(y, groups, n_splits=5, random_state=42):
    """
    Split by capture file rather than by individual window.
    This keeps windows from the same recording together in the same fold.
    """
    df = pd.DataFrame({"y": y, "group": groups})
    group_df = df.groupby("group", as_index=False).agg(y=("y", "first"))

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for train_group_idx, val_group_idx in skf.split(group_df["group"], group_df["y"]):
        train_groups = set(group_df.iloc[train_group_idx]["group"])
        val_groups = set(group_df.iloc[val_group_idx]["group"])

        train_idx = np.where(np.isin(groups, list(train_groups)))[0]
        val_idx = np.where(np.isin(groups, list(val_groups)))[0]
        yield train_idx, val_idx


fold_results = []
all_true = []
all_pred = []

for fold, (train_idx, val_idx) in enumerate(
    make_group_stratified_folds(
        y=y,
        groups=groups,
        n_splits=CONFIG["n_splits"],
        random_state=CONFIG["random_seed"],
    ),
    start=1,
):
    print("=" * 80)
    print(f"Fold {fold}/{CONFIG['n_splits']}")
    print("Train windows:", len(train_idx), "| Val windows:", len(val_idx))
    print("Train groups:", len(np.unique(groups[train_idx])), "| Val groups:", len(np.unique(groups[val_idx])))

    train_ds = IQWindowDataset(X[train_idx], y[train_idx])
    val_ds = IQWindowDataset(X[val_idx], y[val_idx])

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)

    model = IQCNN(num_classes=len(class_names)).to(DEVICE)

    model, history = train_one_fold(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=CONFIG["epochs"],
        learning_rate=CONFIG["learning_rate"],
        weight_decay=CONFIG["weight_decay"],
        device=DEVICE,
    )

    y_true_fold, y_pred_fold = predict_model(model, val_loader, DEVICE)

    acc = accuracy_score(y_true_fold, y_pred_fold)
    f1_macro = f1_score(y_true_fold, y_pred_fold, average="macro")
    f1_weighted = f1_score(y_true_fold, y_pred_fold, average="weighted")

    fold_results.append({
        "fold": fold,
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
    })

    all_true.extend(y_true_fold.tolist())
    all_pred.extend(y_pred_fold.tolist())

    print(f"Fold {fold} accuracy     : {acc:.4f}")
    print(f"Fold {fold} macro F1     : {f1_macro:.4f}")
    print(f"Fold {fold} weighted F1  : {f1_weighted:.4f}")

cv_results_df = pd.DataFrame(fold_results)
cv_results_df


In [ ]:
# 8. Cross-validation summary

print(cv_results_df)
print()
print("Mean accuracy    :", cv_results_df["accuracy"].mean())
print("Std accuracy     :", cv_results_df["accuracy"].std(ddof=1))
print("Mean macro F1    :", cv_results_df["f1_macro"].mean())
print("Mean weighted F1 :", cv_results_df["f1_weighted"].mean())

overall_true = np.array(all_true)
overall_pred = np.array(all_pred)

print("\nOverall classification report across validation folds:")
print(classification_report(overall_true, overall_pred, target_names=class_names))


In [ ]:
# 9. Confusion matrix

cm = confusion_matrix(overall_true, overall_pred)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation="nearest")
ax.set_title("Confusion matrix across all validation folds")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_xticks(np.arange(len(class_names)))
ax.set_yticks(np.arange(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticklabels(class_names)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")

fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


## Train final model on all available windows

After checking the cross-validation results, this trains one final model using all of the windows.

This saved model is the one I would use for later testing on new captures. For ADS-B work, the same structure should still apply once the labels are changed from signal type to aircraft, transmitter, or capture source depending on the experiment.


In [ ]:
# 10. Train the final model on all windows

full_ds = IQWindowDataset(X, y)
full_loader = DataLoader(full_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=0)

final_model = IQCNN(num_classes=len(class_names)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    final_model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)

for epoch in range(1, CONFIG["epochs"] + 1):
    final_model.train()
    running_loss = 0.0
    y_true_epoch, y_pred_epoch = [], []

    for xb, yb in full_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer.zero_grad()
        logits = final_model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xb.size(0)
        preds = torch.argmax(logits, dim=1)
        y_true_epoch.extend(yb.detach().cpu().numpy())
        y_pred_epoch.extend(preds.detach().cpu().numpy())

    epoch_loss = running_loss / len(full_loader.dataset)
    epoch_acc = accuracy_score(y_true_epoch, y_pred_epoch)
    print(f"Final model epoch {epoch:02d}/{CONFIG['epochs']} | loss={epoch_loss:.4f} acc={epoch_acc:.4f}")


In [ ]:
# 11. Save the model and metadata

model_path = CONFIG["artifact_dir"] / "iq_cnn_model.pt"
meta_path = CONFIG["artifact_dir"] / "iq_cnn_metadata.json"

torch.save(final_model.state_dict(), model_path)

metadata = {
    "class_names": class_names,
    "window_len": CONFIG["window_len"],
    "stride": CONFIG["stride"],
    "normalise_per_window": CONFIG["normalise_per_window"],
    "demean": CONFIG["demean"],
    "num_filters_1": CONFIG["num_filters_1"],
    "num_filters_2": CONFIG["num_filters_2"],
    "kernel_size_1": CONFIG["kernel_size_1"],
    "kernel_size_2": CONFIG["kernel_size_2"],
    "fc_dim": CONFIG["fc_dim"],
    "cv_results": cv_results_df.to_dict(orient="records"),
}

with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved model to:", model_path.resolve())
print("Saved metadata to:", meta_path.resolve())


## Inference on a single new capture

This section is for testing one new capture after the model has been trained.

The file is loaded, split into windows, classified window-by-window, and then combined into one overall decision for the capture. For a later live version, this could be adapted to work on a rolling buffer of recent I/Q samples rather than a saved file.


In [ ]:
# 12. Inference helpers

@torch.no_grad()
def predict_windows(model, windows_2ch: np.ndarray, device: str = DEVICE):
    model.eval()
    xb = torch.tensor(windows_2ch, dtype=torch.float32).to(device)
    logits = model(xb)
    probs = torch.softmax(logits, dim=1).cpu().numpy()
    preds = probs.argmax(axis=1)
    return preds, probs


def file_to_windows(path: Path):
    iq = load_iq_file(path)
    windows = []
    for win in generate_windows(iq, CONFIG["window_len"], CONFIG["stride"]):
        if CONFIG["normalise_per_window"]:
            win = normalise_iq(win, demean=CONFIG["demean"])
        windows.append(iq_to_2ch(win))
    if not windows:
        raise ValueError(f"No valid windows produced for {path}")
    return np.stack(windows)


def aggregate_predictions(preds, probs, class_names):
    majority_idx = Counter(preds).most_common(1)[0][0]
    mean_prob = probs.mean(axis=0)
    mean_idx = int(np.argmax(mean_prob))

    return {
        "majority_vote_class": class_names[majority_idx],
        "mean_probability_class": class_names[mean_idx],
        "mean_probability_vector": {class_names[i]: float(mean_prob[i]) for i in range(len(class_names))}
    }